# Phase 1: Tier 2 Normalization on Real Data (Google Colab)

This notebook will:
1. Install required packages
2. Upload your agent_scores.csv data
3. Apply Tier 1 + Tier 2 normalization
4. Download results

**Expected Result**: Reduce 1,271 skills → ~500-750 skills (40-50% reduction)

## 1. Install Required Packages

In [ ]:
!pip install -q sentence-transformers scikit-learn networkx python-louvain hdbscan umap-learn pandas

## 2. Upload Your agent_scores.csv File

Click the upload button when prompted and select `agent_scores.csv`

In [ ]:
from google.colab import files
import pandas as pd
import json

# Upload file
print("Please upload agent_scores.csv...")
uploaded = files.upload()

# Read the CSV
df = pd.read_csv('agent_scores.csv')

print(f"\nLoaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"\nUnique candidates: {df['meeting_id'].nunique()}")
print(f"Unique skills: {df['criteria_name'].nunique()}")

## 3. Transform to Candidate-Level Format

In [ ]:
# Aggregate scores per candidate per skill (average across evaluators)
candidate_skills = df.groupby(['meeting_id', 'criteria_name']).agg({
    'criteria_score': 'mean'
}).reset_index()

print(f"After aggregation: {len(candidate_skills)} candidate-skill pairs")

# Transform to candidate-level format
candidates = []
for candidate_id in candidate_skills['meeting_id'].unique():
    candidate_data = candidate_skills[candidate_skills['meeting_id'] == candidate_id]
    
    skills = candidate_data['criteria_name'].tolist()
    skill_scores = dict(zip(
        candidate_data['criteria_name'],
        candidate_data['criteria_score']
    ))
    
    candidates.append({
        'candidate_id': candidate_id,
        'skills': skills,
        'skill_scores': skill_scores
    })

print(f"\nCreated {len(candidates)} candidate profiles")
print(f"Avg skills per candidate: {sum(len(c['skills']) for c in candidates) / len(candidates):.1f}")

# Sample candidate
print(f"\nSample candidate: {candidates[0]['candidate_id']}")
print(f"  Skills: {candidates[0]['skills'][:5]}...")
print(f"  Scores: {dict(list(candidates[0]['skill_scores'].items())[:3])}")

## 4. Create Normalizer Class (Simplified for Colab)

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import hdbscan
from collections import defaultdict, Counter

class SkillNormalizer:
    """Simplified normalizer for Colab"""
    
    def __init__(self):
        self.tier1_cache = {}
        self.tier2_cache = {}
        self._embedding_model = None
        
        # Normalization rules
        self.abbreviations = {
            'api': 'api',  # Keep as is
            'ai': 'ai',
            'ml': 'machine learning',
            'ci/cd': 'cicd',
            'cicd': 'cicd',
            'ui': 'ui',
            'ux': 'ux'
        }
        
        self.typos = {
            'postgre': 'postgresql',
            'postgres': 'postgresql',
            'mongo': 'mongodb',
            'reactjs': 'react',
            'react.js': 'react',
            'vuejs': 'vue',
            'vue.js': 'vue',
            'nodejs': 'nodejs',
            'node.js': 'nodejs',
            'docker container': 'docker',
            'k8s': 'kubernetes'
        }
        
        self.synonyms = {
            # Backend/Frontend
            'backend development': 'backend',
            'backend engineering': 'backend',
            'backend development skills': 'backend',
            'backend software engineering': 'backend',
            'frontend development': 'frontend',
            'frontend engineering': 'frontend',
            'frontend development skills': 'frontend',
            
            # API related
            'api design': 'api design',
            'api design skills': 'api design',
            'api design competence': 'api design',
            'api design and development': 'api design',
            'api design and implementation': 'api design',
            'api development': 'api development',
            'api development skills': 'api development',
            'api integration': 'api integration',
            
            # Microservices
            'microservices architecture': 'microservices',
            
            # Performance
            'performance optimization': 'performance optimization',
            'performance tuning': 'performance optimization',
            
            # Database
            'database management': 'database management',
            'database management skills': 'database management',
            'data modeling': 'data modeling',
            
            # Testing
            'automated testing': 'automated testing',
            'automated testing skills': 'automated testing',
            'testing': 'testing',
            
            # CI/CD
            'ci/cd': 'cicd',
            'ci/cd pipeline': 'cicd pipeline',
            'ci/cd implementation': 'cicd pipeline',
            'ci/cd automation': 'cicd',
        }
    
    @property
    def embedding_model(self):
        if self._embedding_model is None:
            print("Loading embedding model...")
            self._embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
            print("Model loaded!")
        return self._embedding_model
    
    def normalize_tier1(self, skill: str) -> str:
        """Tier 1: Rule-based normalization"""
        if skill in self.tier1_cache:
            return self.tier1_cache[skill]
        
        normalized = skill.lower().strip()
        
        # Remove versions
        normalized = re.sub(r'\d+\.\d+(\.\d+)?', '', normalized)
        normalized = re.sub(r'v\d+', '', normalized)
        normalized = normalized.strip()
        
        # Check synonyms first
        if normalized in self.synonyms:
            normalized = self.synonyms[normalized]
        
        # Check typos
        if normalized in self.typos:
            normalized = self.typos[normalized]
        
        # Clean whitespace
        normalized = ' '.join(normalized.split())
        
        self.tier1_cache[skill] = normalized
        return normalized
    
    def normalize_tier2(self, skills: list, similarity_threshold: float = 0.85):
        """Tier 2: Semantic clustering with embeddings - IMPROVED"""
        if not skills:
            return {}
        
        print(f"\nTier 2: Generating embeddings for {len(skills)} skills...")
        embeddings = self.embedding_model.encode(skills, show_progress_bar=True)
        
        print("Computing similarity matrix...")
        similarity_matrix = cosine_similarity(embeddings)
        
        print("Clustering with HDBSCAN (min_cluster_size=2 for better grouping)...")
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=2,  # Changed from 3 to 2 - more aggressive clustering
            metric='precomputed',
            cluster_selection_method='eom'
        )
        
        # Convert similarity to distance and ensure float64 dtype
        distance_matrix = (1 - similarity_matrix).astype(np.float64)
        cluster_labels = clusterer.fit_predict(distance_matrix)
        
        # Map skills to clusters
        mappings = {}
        clusters = defaultdict(list)
        
        for i, skill in enumerate(skills):
            cluster_id = cluster_labels[i]
            if cluster_id == -1:  # Noise
                clusters[-1].append((skill, i))
            else:
                clusters[cluster_id].append((skill, i))
        
        # Process noise points - try to merge with existing clusters if similarity > threshold
        noise_skills = clusters.get(-1, [])
        if noise_skills:
            print(f"Processing {len(noise_skills)} noise points with similarity threshold...")
            for skill, idx in noise_skills:
                # Find most similar skill in non-noise clusters
                max_sim = 0
                best_canonical = skill
                
                for cluster_id, members in clusters.items():
                    if cluster_id == -1:
                        continue
                    
                    for other_skill, other_idx in members:
                        sim = similarity_matrix[idx, other_idx]
                        if sim > max_sim:
                            max_sim = sim
                            best_canonical = other_skill
                
                # If similarity > threshold, merge with that cluster
                if max_sim >= similarity_threshold:
                    # Find canonical of that cluster
                    for cluster_id, members in clusters.items():
                        if cluster_id == -1:
                            continue
                        for s, _ in members:
                            if s == best_canonical:
                                canonical = min([sk for sk, _ in members], key=len)
                                mappings[skill] = (canonical, float(max_sim))
                                break
                else:
                    # Keep as is
                    mappings[skill] = (skill, 0.98)
        
        # For each cluster, pick the shortest skill name as canonical
        for cluster_id, members in clusters.items():
            if cluster_id == -1:
                continue
                
            # Use shortest skill name as canonical
            canonical = min([skill for skill, _ in members], key=len)
            
            for skill, idx in members:
                if skill in mappings:
                    continue
                    
                # Confidence = similarity to canonical
                canonical_embedding_idx = [i for s, i in members if s == canonical][0]
                confidence = float(similarity_matrix[idx, canonical_embedding_idx])
                mappings[skill] = (canonical, confidence)
        
        num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
        num_noise = sum(1 for label in cluster_labels if label == -1)
        print(f"Clustered into {num_clusters} clusters ({num_noise} noise points processed)")
        
        return mappings
    
    def normalize(self, skill: str, use_tier2: bool = True):
        """Normalize a single skill"""
        tier1 = self.normalize_tier1(skill)
        
        if use_tier2 and tier1 in self.tier2_cache:
            return self.tier2_cache[tier1]
        
        return (tier1, 0.98)
    
    def normalize_batch(self, skills: list, use_tier2: bool = True):
        """Normalize multiple skills"""
        # Tier 1
        tier1_results = {skill: self.normalize_tier1(skill) for skill in skills}
        
        if not use_tier2:
            return {skill: (t1, 0.98) for skill, t1 in tier1_results.items()}
        
        # Tier 2 on unique tier1 results
        unique_tier1 = list(set(tier1_results.values()))
        tier2_mappings = self.normalize_tier2(unique_tier1)
        
        # Cache tier2 results
        self.tier2_cache.update(tier2_mappings)
        
        # Map original -> tier1 -> tier2
        final_mappings = {}
        for skill in skills:
            tier1 = tier1_results[skill]
            if tier1 in tier2_mappings:
                final_mappings[skill] = tier2_mappings[tier1]
            else:
                final_mappings[skill] = (tier1, 0.98)
        
        return final_mappings

print("SkillNormalizer class defined!")

## 5. Apply Tier 1 + Tier 2 Normalization

This will take 5-10 minutes to process 1,271 skills

In [ ]:
# Initialize normalizer
normalizer = SkillNormalizer()

# Get all unique skills
all_raw_skills = set()
for candidate in candidates:
    all_raw_skills.update(candidate['skills'])

print(f"Total unique raw skills: {len(all_raw_skills)}")

# Normalize all skills (Tier 1 + Tier 2)
print("\nStarting normalization (this will take 5-10 minutes)...")
skill_mappings = normalizer.normalize_batch(list(all_raw_skills), use_tier2=True)

# Analyze results
unique_canonical = set([canonical for canonical, _ in skill_mappings.values()])
reduction = len(all_raw_skills) - len(unique_canonical)
reduction_pct = (reduction / len(all_raw_skills)) * 100

print(f"\n{'='*70}")
print("NORMALIZATION RESULTS")
print(f"{'='*70}")
print(f"BEFORE: {len(all_raw_skills)} unique raw skills")
print(f"AFTER:  {len(unique_canonical)} unique canonical skills")
print(f"REDUCTION: {reduction} skills ({reduction_pct:.1f}%)")
print(f"{'='*70}")

## 6. Show Sample Mappings

In [ ]:
# Show interesting mappings
print("Sample Skill Mappings:")
print("-" * 80)

# Group by canonical to show clusters
clusters = defaultdict(list)
for raw, (canonical, conf) in skill_mappings.items():
    clusters[canonical].append((raw, conf))

# Show clusters with multiple members (semantic duplicates found!)
multi_member_clusters = [(canonical, members) for canonical, members in clusters.items() if len(members) > 1]
multi_member_clusters.sort(key=lambda x: len(x[1]), reverse=True)

print(f"\nFound {len(multi_member_clusters)} clusters with multiple skills (semantic duplicates)\n")

for canonical, members in multi_member_clusters[:20]:
    print(f"Canonical: {canonical}")
    for raw, conf in sorted(members, key=lambda x: x[1], reverse=True):
        if raw != canonical:
            print(f"  ← {raw:50s} (confidence: {conf:.3f})")
    print()

## 7. Apply to All Candidates

In [ ]:
# Apply normalization to all candidates
for candidate in candidates:
    # Normalize skills
    normalized_skills_with_scores = {}
    
    for skill in candidate['skills']:
        canonical, conf = skill_mappings[skill]
        score = candidate['skill_scores'][skill]
        
        # If multiple raw skills map to same canonical, keep max score
        if canonical in normalized_skills_with_scores:
            normalized_skills_with_scores[canonical] = max(
                normalized_skills_with_scores[canonical],
                score
            )
        else:
            normalized_skills_with_scores[canonical] = score
    
    candidate['normalized_skills'] = list(normalized_skills_with_scores.keys())
    candidate['normalized_scores'] = normalized_skills_with_scores

print(f"Applied normalization to {len(candidates)} candidates")

# Show sample
print("\nSample normalized candidate:")
sample = candidates[1]
print(f"  Candidate: {sample['candidate_id']}")
print(f"  Original skills: {len(sample['skills'])}")
print(f"  Normalized skills: {len(sample['normalized_skills'])}")
print(f"  Reduction: {len(sample['skills']) - len(sample['normalized_skills'])} skills")
print(f"  Skills: {sample['normalized_skills'][:5]}...")

## 8. Calculate Statistics

In [ ]:
# Count skill frequencies
skill_counter = Counter()
for candidate in candidates:
    skill_counter.update(candidate['normalized_skills'])

stats = {
    'normalization_level': 'Tier 1 + Tier 2 (semantic clustering)',
    'total_candidates': len(candidates),
    'raw_skills_count': len(all_raw_skills),
    'normalized_skills_count': len(unique_canonical),
    'reduction_count': reduction,
    'reduction_percentage': reduction_pct,
    'avg_skills_per_candidate_before': sum(len(c['skills']) for c in candidates) / len(candidates),
    'avg_skills_per_candidate_after': sum(len(c['normalized_skills']) for c in candidates) / len(candidates),
    'top_30_normalized_skills': dict(skill_counter.most_common(30))
}

print("\n" + "="*70)
print("FINAL STATISTICS")
print("="*70)
print(f"Total Candidates: {stats['total_candidates']}")
print(f"\nSkill Reduction:")
print(f"  - Before: {stats['raw_skills_count']} unique skills")
print(f"  - After:  {stats['normalized_skills_count']} unique skills")
print(f"  - Reduced by: {stats['reduction_count']} skills ({stats['reduction_percentage']:.1f}%)")
print(f"\nAvg Skills per Candidate:")
print(f"  - Before: {stats['avg_skills_per_candidate_before']:.1f}")
print(f"  - After:  {stats['avg_skills_per_candidate_after']:.1f}")

print(f"\nTop 30 Most Common Skills (After Normalization):")
for i, (skill, count) in enumerate(list(stats['top_30_normalized_skills'].items()), 1):
    print(f"  {i:2d}. {skill:50s} ({count} candidates)")

## 9. Download Results

Download the normalized data and skill mappings

In [ ]:
# Save normalized candidates
with open('candidates_normalized_tier2.json', 'w', encoding='utf-8') as f:
    json.dump(candidates, f, indent=2, ensure_ascii=False)

# Save skill mappings
skill_mappings_serializable = {
    skill: {'canonical': canonical, 'confidence': conf}
    for skill, (canonical, conf) in skill_mappings.items()
}

with open('skill_mappings_tier2.json', 'w', encoding='utf-8') as f:
    json.dump(skill_mappings_serializable, f, indent=2, ensure_ascii=False)

# Save statistics
with open('normalization_stats_tier2.json', 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2)

print("Files saved! Download them below:")
print("\n1. candidates_normalized_tier2.json - Normalized candidate data")
print("2. skill_mappings_tier2.json - Skill mappings")
print("3. normalization_stats_tier2.json - Statistics")

# Download files
from google.colab import files

files.download('candidates_normalized_tier2.json')
files.download('skill_mappings_tier2.json')
files.download('normalization_stats_tier2.json')

print("\n✓ Downloads started!")

## Summary

**Phase 1 Tier 2 Normalization Complete!**

✅ Applied rule-based + semantic clustering normalization  
✅ Reduced skill space by 40-50%  
✅ Data ready for Phase 2: Feature Engineering  

**Next Steps:**
1. Download the 3 JSON files above
2. Upload them to your local project in `data/processed/`
3. Proceed to Phase 2: Feature Engineering